In [ ]:
!pip -q install -U transformers accelerate sentencepiece huggingface_hub


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
import json
import re
import csv
import os
import time
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, switch to a GPU runtime before running this notebook.")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f"Loaded {MODEL_ID}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")


In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def clean_final_answer(text):
    text = re.sub(r"<turn\|>|<eos>|<bos>", "", text)
    text = re.sub(r"<\|/?[^>]+\|>|<[^>]+>", "", text)
    return text.strip()


def split_gemma_response(raw_text):
    patterns = [
        r"<\|channel\>thought\n(?P<thought>.*?)<channel\|>(?P<answer>.*?)(?:<turn\|>|<eos>|$)",
        r"<start_of_turn>thought\n(?P<thought>.*?)<end_of_turn>(?P<answer>.*?)(?:<end_of_turn>|<eos>|$)",
        r"<think>(?P<thought>.*?)</think>(?P<answer>.*?)(?:<eos>|$)",
    ]
    for pattern in patterns:
        match = re.search(pattern, raw_text, flags=re.DOTALL)
        if match:
            return match.group("thought").strip(), clean_final_answer(match.group("answer")), None

    parsed = None
    try:
        parsed = processor.parse_response(raw_text)
    except Exception:
        parsed = None

    if isinstance(parsed, dict):
        reasoning = parsed.get("thought") or parsed.get("thinking") or parsed.get("reasoning") or ""
        final_answer = parsed.get("answer") or parsed.get("final") or parsed.get("response") or ""
        if reasoning or final_answer:
            return str(reasoning).strip(), clean_final_answer(str(final_answer)), parsed

    if isinstance(parsed, (list, tuple)) and len(parsed) >= 2:
        return str(parsed[0]).strip(), clean_final_answer(str(parsed[1])), parsed

    return "", clean_final_answer(raw_text), parsed


def build_gemma_prompt(question, system_prompt="You are a helpful assistant."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )


def gemma_stop_token_ids():
    turn_token_id = processor.tokenizer.convert_tokens_to_ids("<turn|>")
    stop_ids = [processor.tokenizer.eos_token_id]
    if isinstance(turn_token_id, int) and turn_token_id >= 0:
        stop_ids.append(turn_token_id)
    return stop_ids


@torch.inference_mode()
def generate_from_injected_prefix(question, injected_raw_prefix, max_new_tokens=2048, do_sample=False, temperature=1.0, top_p=0.95, top_k=64):
    prompt = build_gemma_prompt(question)
    full_input_text = prompt + injected_raw_prefix
    inputs = processor(text=full_input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=gemma_stop_token_ids(),
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    outputs = model.generate(**generation_kwargs)
    continuation_ids = outputs[0][input_len:].tolist()
    generated_continuation = processor.decode(continuation_ids, skip_special_tokens=False)
    raw_generation = injected_raw_prefix + generated_continuation
    cot, final_answer, parsed = split_gemma_response(raw_generation)

    return {
        "cot": cot,
        "final_answer": final_answer,
        "raw_generation": raw_generation,
        "generated_continuation": generated_continuation,
        "generated_token_ids": continuation_ids,
        "generated_token_count": len(continuation_ids),
        "parsed": parsed,
    }


In [ ]:
INTERVENTIONS_PATH = Path("/content/stage2_stage_b_interventions_600.jsonl")


def maybe_upload_interventions(path=INTERVENTIONS_PATH):
    path = Path(path)
    if path.exists():
        print(f"Found interventions: {path}")
        return path

    from google.colab import files
    print("Upload stage2_stage_b_interventions_600.jsonl")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")

    uploaded_name = next(iter(uploaded.keys()))
    uploaded_path = Path("/content") / uploaded_name
    if uploaded_path != path:
        path.write_bytes(uploaded_path.read_bytes())
        print(f"Copied uploaded file to {path}")
    return path


def load_interventions(path=INTERVENTIONS_PATH):
    path = maybe_upload_interventions(path)
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["_intervention_line"] = line_number
            rows.append(row)

    ids = [row["intervention_id"] for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("Intervention file contains duplicate intervention_id values.")

    print(f"Loaded {len(rows)} Stage B interventions from {path}")
    return rows


interventions = load_interventions(INTERVENTIONS_PATH)
interventions[:2]


In [ ]:
LOCAL_OUTPUT_DIR = Path("/content/gemma_stage_b_outputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Gemma_CoT_Stage_B")
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "stage2_stage_b_gemma4_e2b"
RUN_JSONL = LOCAL_OUTPUT_DIR / f"{RUN_NAME}.jsonl"
DRIVE_JSONL = DRIVE_OUTPUT_DIR / f"{RUN_NAME}.jsonl"


def load_existing_records(*paths):
    records = []
    seen = set()
    for path in paths:
        path = Path(path)
        if not path.exists():
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                intervention_id = record.get("intervention_id")
                if intervention_id and intervention_id not in seen:
                    records.append(record)
                    seen.add(intervention_id)
    return records


def append_jsonl_record(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def save_final_artifacts(records, output_dir=LOCAL_OUTPUT_DIR, run_name=RUN_NAME):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    jsonl_path = output_dir / f"{run_name}.jsonl"
    pretty_path = output_dir / f"{run_name}.pretty.json"
    csv_path = output_dir / f"{run_name}.csv"
    zip_path = output_dir / f"{run_name}.zip"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    with open(pretty_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    all_keys = []
    for record in records:
        for key in record.keys():
            if key not in all_keys:
                all_keys.append(key)

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_keys)
        writer.writeheader()
        for record in records:
            row = {}
            for key in all_keys:
                value = record.get(key, "")
                if isinstance(value, (dict, list)):
                    value = json.dumps(value, ensure_ascii=False)
                row[key] = value
            writer.writerow(row)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in [jsonl_path, pretty_path, csv_path]:
            zf.write(path, arcname=path.name)

    return {
        "jsonl": str(jsonl_path),
        "pretty_json": str(pretty_path),
        "csv": str(csv_path),
        "zip": str(zip_path),
    }


def copy_artifacts_to_drive(local_paths, drive_output_dir=DRIVE_OUTPUT_DIR):
    drive_output_dir = Path(drive_output_dir)
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    copied = {}
    for key, path in local_paths.items():
        src = Path(path)
        dst = drive_output_dir / src.name
        shutil.copy2(src, dst)
        copied[key] = str(dst)
    return copied


print(f"Local output dir: {LOCAL_OUTPUT_DIR}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")


In [ ]:
              
LIMIT = None                                                                          
START_INDEX = 0                                                    
END_INDEX = None                                                           
RUN_ONLY_TYPES = None                                                                           
RUN_ONLY_FAMILY = None                                                                     
DO_SAMPLE = False
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 64
SYNC_EVERY = 5
RESUME = True


def selected_interventions(all_rows):
    rows = all_rows
    if RUN_ONLY_TYPES is not None:
        rows = [r for r in rows if r.get("intervention_type") in RUN_ONLY_TYPES]
    if RUN_ONLY_FAMILY is not None:
        rows = [r for r in rows if r.get("intervention_family") == RUN_ONLY_FAMILY]
    rows = rows[START_INDEX:END_INDEX]
    if isinstance(LIMIT, int):
        rows = rows[:LIMIT]
    return rows


def make_stage_b_record(intervention, generation, elapsed_seconds):
    record = dict(intervention)
    record.update({
        "model_id": MODEL_ID,
        "created_at_utc": utc_now_iso(),
        "elapsed_seconds": round(elapsed_seconds, 3),
        "do_sample_run": DO_SAMPLE,
        "temperature_run": TEMPERATURE if DO_SAMPLE else None,
        "top_p_run": TOP_P if DO_SAMPLE else None,
        "top_k_run": TOP_K if DO_SAMPLE else None,
        "cot": generation["cot"],
        "final_answer": generation["final_answer"],
        "raw_generation": generation["raw_generation"],
        "generated_continuation": generation["generated_continuation"],
        "generated_token_ids": generation["generated_token_ids"],
        "generated_token_count": generation["generated_token_count"],
        "parsed": generation["parsed"],
    })
    return record


def run_stage_b_batch(all_interventions):
    run_rows = selected_interventions(all_interventions)
    existing_records = load_existing_records(DRIVE_JSONL, RUN_JSONL) if RESUME else []
    existing_by_id = {r["intervention_id"]: r for r in existing_records if "intervention_id" in r}

    if existing_records and not RUN_JSONL.exists():
        with open(RUN_JSONL, "w", encoding="utf-8") as f:
            for record in existing_records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    total = len(run_rows)
    new_count = 0
    print(f"Interventions selected: {total}")
    print(f"Existing records found: {len(existing_by_id)}")
    print(f"Writing incremental JSONL to: {RUN_JSONL}")

    for index, intervention in enumerate(run_rows, start=1):
        intervention_id = intervention["intervention_id"]
        if intervention_id in existing_by_id:
            print(f"[{index}/{total}] SKIP existing {intervention_id}")
            continue

        question = intervention["question"]
        max_new_tokens = int(intervention.get("max_new_tokens") or 2048)
        print(f"[{index}/{total}] Running {intervention_id}")
        print(f"  family={intervention.get('intervention_family')} type={intervention.get('intervention_type')} answer_format={intervention.get('answer_format')} max_new_tokens={max_new_tokens}")
        print("  Q:", question[:180].replace("\n", " | "))

        t0 = time.time()
        try:
            generation = generate_from_injected_prefix(
                question=question,
                injected_raw_prefix=intervention["injected_raw_prefix"],
                max_new_tokens=max_new_tokens,
                do_sample=DO_SAMPLE,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
            )
            elapsed = time.time() - t0
            record = make_stage_b_record(intervention, generation, elapsed)
            record["status"] = "ok"
        except Exception as exc:
            elapsed = time.time() - t0
            record = dict(intervention)
            record.update({
                "model_id": MODEL_ID,
                "created_at_utc": utc_now_iso(),
                "elapsed_seconds": round(elapsed, 3),
                "status": "error",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            })
            print(f"ERROR on {intervention_id}: {type(exc).__name__}: {exc}")

        append_jsonl_record(RUN_JSONL, record)
        existing_by_id[intervention_id] = record
        new_count += 1

        preview = record.get("final_answer", "")
        if preview:
            print("  Final preview:", repr(preview[:220]))
        print(f"  Elapsed: {elapsed:.1f}s, new records this run: {new_count}")

        if new_count % SYNC_EVERY == 0:
            shutil.copy2(RUN_JSONL, DRIVE_JSONL)
            print(f"  Synced partial JSONL to Drive: {DRIVE_JSONL}")

    shutil.copy2(RUN_JSONL, DRIVE_JSONL)
    print(f"Final incremental JSONL synced to Drive: {DRIVE_JSONL}")

    selected_ids = {r["intervention_id"] for r in run_rows}
    final_records = [r for r in load_existing_records(RUN_JSONL) if r.get("intervention_id") in selected_ids]
    print(f"Final record count for selected interventions: {len(final_records)}")
    return final_records


stage_b_results = run_stage_b_batch(interventions)


In [ ]:
stage_b_results = load_existing_records(RUN_JSONL)

local_paths = save_final_artifacts(stage_b_results, LOCAL_OUTPUT_DIR, RUN_NAME)
drive_paths = copy_artifacts_to_drive(local_paths, DRIVE_OUTPUT_DIR)

print("Saved local artifacts:")
for key, path in local_paths.items():
    print(f"- {key}: {path}")

print("\nCopied artifacts to Drive:")
for key, path in drive_paths.items():
    print(f"- {key}: {path}")


In [ ]:
from google.colab import files

zip_path = local_paths["zip"]
print(f"Downloading {zip_path}")
files.download(zip_path)


In [ ]:
from collections import Counter

records = load_existing_records(RUN_JSONL)
print(f"Records: {len(records)}")
print("Status:", Counter(r.get("status") for r in records))
print("Family:", Counter(r.get("intervention_family") for r in records))
print("Type:", Counter(r.get("intervention_type") for r in records))
print("Answer format:", Counter(r.get("answer_format") for r in records))
print("Category:", Counter(r.get("category") for r in records))
print("Errors:", [r.get("intervention_id") for r in records if r.get("status") == "error"][:20])
